# Cross-Validation and Hyperparameter Tuning

In this notebook, we summarize our analysis of cross-validation and hyperparameter tuning for the kNN Regressor model. We also demonstrate the implementation of the Python script `kNN_script.py`.

Suppose we are interested in evaluating the cross-validation error scores for a particular prediction window. Our method performs cross-validation in a rolling fashion: for each day in the prediction window, we set the training window to the prior 60 days. We then obtain the predicted value for that day, move to the next day, and repeat the process by shifting the training window accordingly. At the end, we return all predicted values for the chosen hyperparameter combination.

## Documentation of `kNN_script.py`
We have developed a Python script named `kNN_script.py` that allows the user to specify hyperparameter combinations, the training window length, and the start and duration of the prediction window. The script defines a class `kNN_Cross_Validation` with the following main features:

- The constructor accepts the arguments `pca_comps: List[int]`, `n_nbrs: List[int]`, `days_to_train_on: int`, `mode: str`, `feature_selection: str`, `with_confidence_interval: bool`, and `no_iters: int`. 
- By default, the best-performing hyperparameters are set:`pca_comps = [35]` and `n_nbrs = [5]`. 
- The training window length can be customized using the `days_to_train_on` parameter (default: 60 days).
- The argument `mode` determines which dataframe to use:
    - `Validation` runs on data from 2019 to 2022 (default setting).
    - `Testing` runs on data from 2019 to 2023.
- The argument `feature_selection` determines which feature to use:
    - `spd_cubed_div_temp` runs on the feature (wind speed)$^3$ divided by the temperature in Kelvin at each farm (default setting).
    - `speed_cubed` runs on the feature (wind speed)$^3$ at each farm.
    - `speed` runs on the feature wind speed at each farm.
- The boolean parameter `with_confidence_interval` determines if we want to calculate confidence interval for different estimators. When this is set to `True`:
    - For each day in the testing window, we resample half of the training datapoints with replacement (e.g. if `days_to_train_on` equals 60, we have $30\times 24$ datapoints choosen from the prior 60 day window for the testing day) and run the model `no_iters` times.
    - `no_iters` determines the number of iterations on each day of the training window.
    - These confidence intervals are stored in the attributes `pred_conf_intervals`, `mape_conf_intervals`, `r2_conf_intervals`, and `mae_conf_intervals`.
    


#### Key Methods:
- `input()`: Prompts the user to provide the validation or testing window.
- `manual_input()`: Lets the user to input validation or testing window (same functionality as `input()` without interactive prompts).
- `run()`: loads data, validates input, runs kNN forecasting, and computes evaluation metrics. Accepts a boolean argument `display` that determines if the plots are automatically displayed.
#### Behaviour:
- If only a single `(pca_comp, n_nbr)` pair was given to the constructor (default setting), it runs the model with that configuration and returns predictions, scores, and a `plotly.graph_objects.Figure` object. The plot is automatically displayed (by default).
- If multiple hyperparameter combinations are provided, it performs a manual grid search, runs predictions for each combination, and displays plots (by dafault) for the best-performing models based on MAPE, R² and MAE.
#### Modeling Approach:
- The kNN model is trained on the engineered feature: (wind speed)$^3$ divided by temperature (in Kelvin) for each wind farm.
- Each prediction is based on a rolling 60-day training window.

##### ⚠️ NOTE: 
This script does not handle real-time forecasting. The prediction window must fall strictly within `2019-03-02` (reserving at least 60 days for training) to `2023-12-31` (end of the validation/testing dataset).


## Example of execution of the script and error scores:

In [1]:
from kNN_script import kNN_Cross_Validation

#### Instantiate a `kNN_Cross_Validation` class object with default constructor.

In [2]:
cv = kNN_Cross_Validation()
cv.input()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2019-03-02 and 2022-12-31.

You have entered: 2022-01-01.

Please enter forecast window (in days). It should be an integer between 1 or 365.

Your forecast window is days between: 2022-01-01 and 2022-06-30



In [ ]:
cv.run()

Cross validation running on your prediction window...



#### Instantiate a `kNN_Cross_Validation` class object with customized constructor parameters:

In [7]:
cv_multiple = kNN_Cross_Validation([10, 20, 25, 30, 35], [5, 10, 20, 30, 50], days_to_train_on=100)
cv_multiple.input()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2019-04-11 and 2022-12-31.

You have entered: 2022-01-01.

Please enter forecast window (in days). It should be an integer between 1 or 365.

Your forecast window is days between: 2022-01-01 and 2022-06-30



In [8]:
cv_multiple.run()

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.161, cv_R2 = 0.890, MAE = 4006.6.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.156, cv_R2 = 0.892, MAE = 3943.4.
Prediction done with: PCA comps = 10, knn nbr = 20. Error scores: cv_mape = 0.153, cv_R2 = 0.895, MAE = 3903.5.
Prediction done with: PCA comps = 10, knn nbr = 30. Error scores: cv_mape = 0.151, cv_R2 = 0.895, MAE = 3908.4.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.151, cv_R2 = 0.893, MAE = 3953.8.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.152, cv_R2 = 0.897, MAE = 3902.7.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.145, cv_R2 = 0.901, MAE = 3787.6.
Prediction done with: PCA comps = 20, knn nbr = 20. Error scores: cv_mape = 0.144, cv_R2 = 0.899, MAE = 3828.5.
Prediction done with: PCA comps = 20, knn nbr = 30.

#### Instantiate a `kNN_Cross_Validation` object with customized feature selection.

In [3]:
knn_feature = kNN_Cross_Validation([10, 20, 25, 30, 35], [5, 10, 20, 30, 50],feature_selection="speed")
knn_feature.input()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2019-03-02 and 2022-12-31.

You have entered: 2022-01-01.

Please enter forecast window (in days). It should be an integer between 1 or 365.

Your forecast window is days between: 2022-01-01 and 2022-06-30



In [4]:
knn_feature.run()

Cross validation running on your prediction window...

Prediction done with: PCA comps = 10, knn nbr = 5. Error scores: cv_mape = 0.218, cv_R2 = 0.841, MAE = 4989.4.
Prediction done with: PCA comps = 10, knn nbr = 10. Error scores: cv_mape = 0.229, cv_R2 = 0.834, MAE = 5118.3.
Prediction done with: PCA comps = 10, knn nbr = 20. Error scores: cv_mape = 0.243, cv_R2 = 0.819, MAE = 5356.6.
Prediction done with: PCA comps = 10, knn nbr = 30. Error scores: cv_mape = 0.252, cv_R2 = 0.812, MAE = 5473.5.
Prediction done with: PCA comps = 10, knn nbr = 50. Error scores: cv_mape = 0.267, cv_R2 = 0.796, MAE = 5667.5.
Prediction done with: PCA comps = 20, knn nbr = 5. Error scores: cv_mape = 0.229, cv_R2 = 0.831, MAE = 5134.4.
Prediction done with: PCA comps = 20, knn nbr = 10. Error scores: cv_mape = 0.244, cv_R2 = 0.821, MAE = 5349.8.
Prediction done with: PCA comps = 20, knn nbr = 20. Error scores: cv_mape = 0.254, cv_R2 = 0.813, MAE = 5463.3.
Prediction done with: PCA comps = 20, knn nbr = 30.

#### Instantiate a `kNN_Cross_Validation` object with confidence interval calculations
To reduce runtime, we only allow to run confidence interval calculation with single hyperparameters pair. 

In [3]:
knn_ci = kNN_Cross_Validation(with_confidence_interval=True, days_to_train_on= 180, no_iters=30)
knn_ci.input()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2019-06-30 and 2022-12-31.

You have entered: 2022-01-01.

Please enter forecast window (in days). It should be an integer between 1 or 365.

Your forecast window is days between: 2022-01-01 and 2022-06-30



In [4]:
knn_ci.run()

Cross validation running on your prediction window...

Note: Running with iterations... This may take longer time.
Running with iterations... This may take longer time.
Error score: mean MAPE = 0.140, mean R2= 0.909, mean MAE = 3739.32482504604.
Confidence interval for MAPE: (np.float64(0.13453517374899956), np.float64(0.1468145439735744)).
Confidence interval for R2: (np.float64(0.9037438394668624), np.float64(0.9141416878629185)).
Confidence interval for MAE: (np.float64(3601.3729834254145), np.float64(3838.7634530386736)).


#### Running with small testing window

In [2]:
knn_ci = kNN_Cross_Validation(with_confidence_interval=True, days_to_train_on= 180, no_iters=30)
knn_ci.manual_input('2022-01-01', 15)


Your forecast window is days between: 2022-01-01 and 2022-01-15



In [3]:
knn_ci.run()

Cross validation running on your prediction window...

Note: Running with iterations... This may take longer time.
Running with iterations... This may take longer time.
Error score: mean MAPE = 0.165, mean R2= 0.933, mean MAE = 3611.385777777778.
Confidence interval for MAPE: (np.float64(0.1312354410709757), np.float64(0.1967017815685614)).
Confidence interval for R2: (np.float64(0.9049442173012112), np.float64(0.9491939673012988)).
Confidence interval for MAE: (np.float64(3050.138666666666), np.float64(4340.635999999999)).


## Running on the Final Testing Dataset

In [11]:
from kNN_script import kNN_Cross_Validation

In [17]:
knn_test = kNN_Cross_Validation(mode="Testing")
knn_test.manual_input("2023-01-01",365)
knn_test.run()


Your forecast window is days between: 2023-01-01 and 2023-12-31

Cross validation running on your prediction window...



Running the test set with confidence interval calculation. 

In [14]:
knn_test = kNN_Cross_Validation( mode="Testing", with_confidence_interval=True)
knn_test.manual_input("2023-01-01",365)
knn_test.run()


Your forecast window is days between: 2023-01-01 and 2023-12-31

Cross validation running on your prediction window...

Note: Running with iterations... This may take longer time.
Running with iterations... This may take longer time.
Error score: mean MAPE = 0.218, mean R2= 0.872, mean MAE = 4307.238696876712.
Confidence interval for MAPE: (np.float64(0.21113522196499246), np.float64(0.2264429344641439)).
Confidence interval for R2: (np.float64(0.8658137236253158), np.float64(0.8783125765580274)).
Confidence interval for MAE: (np.float64(4166.416074383561), np.float64(4422.739945616438)).
